# **Image Novelty Evaluation - Stable Diffusion XL Model**

Evaluates how prompt abstraction level affects image-text alignment (CLIP score), novelty vs. real-world images, and intra-prompt diversity.

# **Import Libraries**

In [ ]:
import torch
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
from sklearn.metrics.pairwise import cosine_similarity
from datasets import load_dataset
import scipy.stats as stats
import seaborn as sns
import matplotlib.pyplot as plt
import os
import itertools
from diffusers import DiffusionPipeline
from transformers import CLIPProcessor, CLIPModel
import random
from statsmodels.stats.multicomp import pairwise_tukeyhsd


Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


# **Setup**

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

seed = 42
torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)

output_dir = "generated_images"
os.makedirs(output_dir, exist_ok=True)

print("Device:", device)


Device: cuda


# **Load Stable Diffusion XL Model**

In [ ]:
pipe = DiffusionPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype=torch.float16,
    use_safetensors=True
)
pipe.to(device)



print("Model loaded.")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model_index.json:   0%|          | 0.00/609 [00:00<?, ?B/s]

Fetching 19 files:   0%|          | 0/19 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

Model loaded.


# **Define Three Levels of Prompt Abstraction**

- **Level 1**: Realistic, grounded scenes
- **Level 2**: Surreal, hybrid imagery
- **Level 3**: Abstract/philosophical concepts

> Note: All 5 prompts per level are unique (duplicates from original notebook removed).

In [ ]:
prompts = {
    "level1": [
        "A cinematic photograph of an old fisherman standing on a wooden dock at sunrise, wearing a weathered blue raincoat and rubber boots, holding a fishing net full of silver fish, with seagulls flying overhead, gentle waves reflecting golden sunlight, distant mountains covered in light morning mist, highly detailed textures, natural lighting, shallow depth of field",
        "Ultra-detailed street photography of a busy Tokyo market at night, neon signs in Japanese characters, vendors selling colorful seafood and fresh vegetables, people holding transparent umbrellas under light rain, reflections on wet asphalt, soft cinematic lighting, 85mm lens, high dynamic range",
        "A realistic medieval blacksmith workshop interior, glowing forge with bright orange sparks, muscular blacksmith hammering a sword on an anvil, stone walls covered in tools, wooden beams overhead, smoke particles in the air, warm dramatic lighting, hyper-detailed textures",
        "Professional wildlife photography of a snow leopard walking across a rocky Himalayan cliff, wind blowing through its fur, distant snowy peaks under a cloudy sky, dramatic natural lighting, ultra-high detail, National Geographic style",
        "Interior of a cozy vintage library café, wooden bookshelves filled with old books, warm yellow lamps, a woman reading by the window with coffee steam rising, rain outside the glass, soft cinematic atmosphere, highly realistic rendering"
    ],
    "level2": [
        "A transparent glass elephant filled with glowing bioluminescent jellyfish, walking through a flooded Renaissance cathedral, sunlight refracting through stained glass windows, reflections on water, floating candles, cinematic volumetric lighting, ultra-detailed textures",
        "A futuristic city built inside the branches of a colossal ancient tree, flying vehicles shaped like origami birds, waterfalls flowing between wooden skyscrapers, holographic advertisements in the air, children riding mechanical deer, hyper-realistic, ultra-detailed",
        "An astronaut wearing a Victorian-era suit made of brass and velvet, standing inside a giant hourglass where galaxies replace sand, planets orbiting around him, cosmic dust particles glowing, dramatic chiaroscuro lighting, high-resolution detail",
        "A surreal underwater library where books float like jellyfish, glowing ink flowing from open pages, mermaids wearing modern business suits typing on coral keyboards, beams of sunlight piercing deep ocean water, ultra-realistic textures",

        "A mechanical heart made of clockwork gears and living vines pumping liquid starlight, surrounded by hovering anatomical diagrams written in Sanskrit, bioluminescent fog, floating copper instruments, ultra-detailed hyper-realistic rendering"
    ],
    "level3": [
        "A visual representation of time arguing with gravity inside a collapsing cathedral made of mathematical equations, floating fragments of shattered clocks transforming into birds, light bending into impossible geometric patterns, hyper-detailed, surreal, cinematic realism",
        "The concept of silence rendered as a physical landscape, where sound waves have solidified into crystalline formations, memories fossilized in translucent amber, an infinite recursive horizon stretching into abstraction, hyper-detailed surrealism",
        "A portrait of entropy wearing a crown made of melting stars, standing on a staircase that spirals into infinity, shadows detaching from objects and forming independent beings, highly detailed, impossible physics, dramatic lighting",
        "A courtroom where abstract concepts like justice, fear, and probability appear as physical entities made of light and glass, debating in a space where the floor is a flowing ocean of numbers, ultra-realistic textures, cinematic atmosphere",
        "A universe dreaming about its own creation, galaxies forming brush strokes in a cosmic painting, mathematical constants floating as luminous symbols, reality folding into recursive mirrors, hyper-detailed surreal visualization"
    ]
}

# Verify no duplicates
for level, plist in prompts.items():
    assert len(set(plist)) == len(plist), f"Duplicate prompts found in {level}!"
    print(f"{level}: {len(plist)} unique prompts ✓")


level1: 5 unique prompts ✓
level2: 5 unique prompts ✓
level3: 5 unique prompts ✓


# **Generate Images**

8 seeds per prompt × 5 prompts × 3 levels = **120 images total**

> FIX: Increased from 4 seeds to 8 seeds per prompt for adequate statistical power (n=40 per level).

In [ ]:
images_data = []
NUM_SEEDS = 8

for level, prompt_list in prompts.items():
    for prompt in prompt_list:
        for i in range(NUM_SEEDS):
            generator = torch.Generator(device).manual_seed(seed + i)

            image = pipe(
                prompt,
                num_inference_steps=30,
                guidance_scale=7.5,
                generator=generator
            ).images[0]

            filename = f"{level}_{prompt[:20].replace(' ','_')}_{i}.png"
            path = os.path.join(output_dir, filename)
            image.save(path)

            images_data.append({
                "level": level,
                "prompt": prompt,
                "path": path
            })

print("Total images generated:", len(images_data))
# Expected: 120 (3 levels × 5 prompts × 8 seeds)


  0%|          | 0/30 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/diffusers/pipelines/stable_diffusion_xl/pipeline_stable_diffusion_xl.py:748: FutureWarning: `upcast_vae` is deprecated and will be removed in version 1.0.0. `upcast_vae` is deprecated. Please use `pipe.vae.to(torch.float32)`. For more details, please refer to: https://github.com/huggingface/diffusers/pull/12619#issue-3606633695.
  deprecate(


  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

...

  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

Total images generated: 120


# **Load CLIP Model**

> FIX: Upgraded from `clip-vit-base-patch32` to `clip-vit-large-patch14` — the standard model used in SDXL evaluation papers and by Stability AI during SDXL training.

In [ ]:

CLIP_MODEL_ID = "openai/clip-vit-large-patch14"

clip_model = CLIPModel.from_pretrained(CLIP_MODEL_ID).to(device)
clip_processor = CLIPProcessor.from_pretrained(CLIP_MODEL_ID)

print(f"CLIP model loaded: {CLIP_MODEL_ID}")


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/590 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-large-patch14
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

CLIP model loaded: openai/clip-vit-large-patch14


# **Define Evaluation Functions**

In [11]:
def compute_clip_score(image_path, text):
    """Compute cosine similarity between image and text embeddings (CLIP score)."""
    image = Image.open(image_path).convert("RGB")

    inputs = clip_processor(
        text=[text],
        images=image,
        return_tensors="pt",
        padding=True
    ).to(device)

    with torch.no_grad():
        outputs = clip_model(**inputs)
        image_emb = outputs.image_embeds
        text_emb = outputs.text_embeds

    image_emb = image_emb / image_emb.norm(dim=-1, keepdim=True)
    text_emb = text_emb / text_emb.norm(dim=-1, keepdim=True)

    return (image_emb @ text_emb.T).item()


In [14]:
def get_image_embedding(image_path):
    image = Image.open(image_path).convert("RGB")
    inputs = clip_processor(images=image, return_tensors="pt").to(device)
    with torch.no_grad():
        emb = clip_model.get_image_features(**inputs)

    if hasattr(emb, 'pooler_output'):
        emb = emb.pooler_output
    emb = emb / emb.norm(dim=-1, keepdim=True)
    return emb.squeeze(0).cpu().numpy()


# **Load MS-COCO Baseline for Novelty Estimation**

> FIX: Replaced CIFAR-10 (32×32 low-res photos) with MS-COCO (high-resolution, photorealistic, diverse scenes). CIFAR-10 is an inappropriate baseline for SDXL outputs and artificially inflates novelty scores. MS-COCO is the standard reference dataset in image generation evaluation literature.

In [15]:
BASELINE_SIZE = 2000

print(f"Loading {BASELINE_SIZE} MS-COCO images as baseline...")
coco = load_dataset("detection-datasets/coco", split="train", streaming=True)

baseline_embs = []
for item in tqdm(itertools.islice(coco, BASELINE_SIZE), total=BASELINE_SIZE):
    try:
        image = item["image"].convert("RGB")
        inputs = clip_processor(images=image, return_tensors="pt").to(device)
        with torch.no_grad():
            emb = clip_model.get_image_features(**inputs)

        # FIX: Handle both tensor and object return types
        if hasattr(emb, 'pooler_output'):
            emb = emb.pooler_output
        # emb is now guaranteed to be a plain tensor
        emb = emb / emb.norm(dim=-1, keepdim=True)
        baseline_embs.append(emb.squeeze(0).cpu().numpy())
    except Exception as e:
        print(f"Skipping image: {e}")

baseline_embs = np.vstack(baseline_embs)
print("Baseline shape:", baseline_embs.shape)

Loading 2000 MS-COCO images as baseline...


Resolving data files:   0%|          | 0/40 [00:00<?, ?it/s]

100%|██████████| 2000/2000 [03:56<00:00,  8.46it/s]

Baseline shape: (2000, 768)


# **Compute CLIP Scores and Novelty**

In [16]:
print("Computing CLIP scores and novelty for all images...")

for item in tqdm(images_data):
    # CLIP score: image-text alignment
    item["clip_score"] = compute_clip_score(item["path"], item["prompt"])

    # Novelty: 1 - mean cosine similarity to baseline
    emb = get_image_embedding(item["path"]).reshape(1, -1)
    sim = cosine_similarity(emb, baseline_embs)
    item["approx_novelty"] = 1 - np.mean(sim)

df = pd.DataFrame(images_data)
print("\nDataframe shape:", df.shape)
print(df.head())


Computing CLIP scores and novelty for all images...


100%|██████████| 120/120 [00:29<00:00,  4.02it/s]


Dataframe shape: (120, 5)
    level                                             prompt  \
0  level1  A cinematic photograph of an old fisherman sta...   
1  level1  A cinematic photograph of an old fisherman sta...   
2  level1  A cinematic photograph of an old fisherman sta...   
3  level1  A cinematic photograph of an old fisherman sta...   
4  level1  A cinematic photograph of an old fisherman sta...   

                                                path  clip_score  \
0  generated_images/level1_A_cinematic_photogra_0...    0.299102   
1  generated_images/level1_A_cinematic_photogra_1...    0.290435   
2  generated_images/level1_A_cinematic_photogra_2...    0.294176   
3  generated_images/level1_A_cinematic_photogra_3...    0.312563   
4  generated_images/level1_A_cinematic_photogra_4...    0.313927   

   approx_novelty  
0        0.464302  
1        0.489320  
2        0.469204  
3        0.485157  
4        0.468799  


# **Compute Intra-Prompt Diversity**

In [17]:
print("Computing diversity per prompt...")

diversity_per_prompt = {}

for prompt in df["prompt"].unique():
    group = df[df["prompt"] == prompt]
    embeddings = np.vstack([get_image_embedding(p) for p in group["path"]])

    sim_matrix = cosine_similarity(embeddings)
    np.fill_diagonal(sim_matrix, np.nan)

    diversity_per_prompt[prompt] = 1 - np.nanmean(sim_matrix)

df["prompt_diversity"] = df["prompt"].map(diversity_per_prompt)
print("Diversity computed for", len(diversity_per_prompt), "prompts.")


Computing diversity per prompt...
Diversity computed for 15 prompts.


# **Results Summary**

In [18]:
summary = df.groupby("level").agg({
    "clip_score": ["mean", "std"],
    "prompt_diversity": ["mean", "std"],
    "approx_novelty": ["mean", "std"]
})

print("\n===== SUMMARY (Mean ± Std) =====\n")
print(summary)

def compute_ci(series):
    return stats.t.interval(
        0.95,
        len(series) - 1,
        loc=np.mean(series),
        scale=stats.sem(series)
    )

print("\n===== 95% Confidence Intervals =====")
for metric in ["clip_score", "prompt_diversity", "approx_novelty"]:
    print(f"\n{metric}:")
    for level in ["level1", "level2", "level3"]:
        ci = compute_ci(df[df["level"] == level][metric])
        mean = df[df["level"] == level][metric].mean()
        print(f"  {level}: {mean:.4f}  95% CI [{ci[0]:.4f}, {ci[1]:.4f}]")



===== SUMMARY (Mean ± Std) =====

       clip_score           prompt_diversity           approx_novelty  \
             mean       std             mean       std           mean   
level                                                                   
level1   0.320479  0.023008         0.042229  0.009047       0.504067   
level2   0.316747  0.032789         0.101950  0.031751       0.540303   
level3   0.288630  0.027933         0.135377  0.066466       0.556954   

                  
             std  
level             
level1  0.021751  
level2  0.036375  
level3  0.038133  

===== 95% Confidence Intervals =====

clip_score:
  level1: 0.3205  95% CI [0.3131, 0.3278]
  level2: 0.3167  95% CI [0.3063, 0.3272]
  level3: 0.2886  95% CI [0.2797, 0.2976]

prompt_diversity:
  level1: 0.0422  95% CI [0.0393, 0.0451]
  level2: 0.1020  95% CI [0.0918, 0.1121]
  level3: 0.1354  95% CI [0.1141, 0.1566]

approx_novelty:
  level1: 0.5041  95% CI [0.4971, 0.5110]
  level2: 0.5403  95% CI [0.528

# **One-Way ANOVA**

In [19]:
print("===== ONE-WAY ANOVA =====\n")
for metric in ["clip_score", "prompt_diversity", "approx_novelty"]:
    f, p = stats.f_oneway(
        df[df.level == "level1"][metric],
        df[df.level == "level2"][metric],
        df[df.level == "level3"][metric]
    )
    sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"
    print(f"{metric}: F={f:.4f}, p={p:.4f} {sig}")


===== ONE-WAY ANOVA =====

clip_score: F=15.2535, p=0.0000 ***
prompt_diversity: F=48.5166, p=0.0000 ***
approx_novelty: F=26.9957, p=0.0000 ***


# **Post-Hoc Analysis: Tukey HSD**

> Added pairwise comparisons — required for academic reporting after significant ANOVA.

In [20]:
print("===== TUKEY HSD POST-HOC TEST =====\n")
for metric in ["clip_score", "prompt_diversity", "approx_novelty"]:
    print(f"\n--- {metric} ---")
    result = pairwise_tukeyhsd(df[metric], df["level"])
    print(result.summary())


===== TUKEY HSD POST-HOC TEST =====


--- clip_score ---
Multiple Comparison of Means - Tukey HSD, FWER=0.05 
group1 group2 meandiff p-adj   lower   upper  reject
----------------------------------------------------
level1 level2  -0.0037 0.8247 -0.0187  0.0112  False
level1 level3  -0.0318    0.0 -0.0468 -0.0169   True
level2 level3  -0.0281 0.0001 -0.0431 -0.0132   True
----------------------------------------------------

--- prompt_diversity ---
Multiple Comparison of Means - Tukey HSD, FWER=0.05
group1 group2 meandiff p-adj lower  upper  reject
-------------------------------------------------
level1 level2   0.0597   0.0  0.037 0.0825   True
level1 level3   0.0931   0.0 0.0704 0.1159   True
level2 level3   0.0334 0.002 0.0107 0.0562   True
-------------------------------------------------

--- approx_novelty ---
Multiple Comparison of Means - Tukey HSD, FWER=0.05
group1 group2 meandiff p-adj   lower  upper  reject
---------------------------------------------------
level1 level2 

# **Effect Sizes: Cohen's d**

In [21]:
def cohens_d(a, b):
    """Cohen's d effect size between two groups."""
    pooled_std = np.sqrt((np.std(a, ddof=1)**2 + np.std(b, ddof=1)**2) / 2)
    return (np.mean(a) - np.mean(b)) / pooled_std

print("===== COHEN'S d EFFECT SIZES =====\n")
pairs = [("level1", "level2"), ("level1", "level3"), ("level2", "level3")]
for metric in ["clip_score", "approx_novelty", "prompt_diversity"]:
    print(f"\n{metric}:")
    for a, b in pairs:
        d = cohens_d(df[df.level == a][metric], df[df.level == b][metric])
        magnitude = "large" if abs(d) > 0.8 else "medium" if abs(d) > 0.5 else "small"
        print(f"  {a} vs {b}: d = {d:.3f} ({magnitude})")


===== COHEN'S d EFFECT SIZES =====


clip_score:
  level1 vs level2: d = 0.132 (small)
  level1 vs level3: d = 1.245 (large)
  level2 vs level3: d = 0.923 (large)

approx_novelty:
  level1 vs level2: d = -1.209 (large)
  level1 vs level3: d = -1.704 (large)
  level2 vs level3: d = -0.447 (small)

prompt_diversity:
  level1 vs level2: d = -2.558 (large)
  level1 vs level3: d = -1.964 (large)
  level2 vs level3: d = -0.642 (medium)


# **Correlation Matrix**

In [22]:
print("===== CORRELATION MATRIX =====\n")
corr = df[["clip_score", "approx_novelty", "prompt_diversity"]].corr()
print(corr.round(4))

plt.figure(figsize=(6, 4))
sns.heatmap(corr, annot=True, fmt=".3f", cmap="coolwarm", center=0)
plt.title("Metric Correlation Matrix")
plt.tight_layout()
plt.savefig("results_correlation.png", dpi=150)
plt.close()
print("Saved: results_correlation.png")


===== CORRELATION MATRIX =====

                  clip_score  approx_novelty  prompt_diversity
clip_score            1.0000         -0.0390           -0.1389
approx_novelty       -0.0390          1.0000            0.6872
prompt_diversity     -0.1389          0.6872            1.0000
Saved: results_correlation.png


# **Bootstrap Confidence Intervals**

In [23]:
def bootstrap_mean(series, n=1000):
    """Bootstrap estimate of mean and standard error."""
    means = [np.mean(np.random.choice(series, size=len(series), replace=True)) for _ in range(n)]
    return np.mean(means), np.std(means)

print("===== BOOTSTRAP ESTIMATES =====\n")
for metric in ["clip_score", "approx_novelty", "prompt_diversity"]:
    print(f"\n{metric}:")
    for level in ["level1", "level2", "level3"]:
        mean, std = bootstrap_mean(df[df.level == level][metric].values)
        print(f"  {level}: Mean={mean:.4f}, Bootstrap SE={std:.4f}")


===== BOOTSTRAP ESTIMATES =====


clip_score:
  level1: Mean=0.3206, Bootstrap SE=0.0036
  level2: Mean=0.3170, Bootstrap SE=0.0048
  level3: Mean=0.2885, Bootstrap SE=0.0044

approx_novelty:
  level1: Mean=0.5041, Bootstrap SE=0.0035
  level2: Mean=0.5403, Bootstrap SE=0.0057
  level3: Mean=0.5568, Bootstrap SE=0.0060

prompt_diversity:
  level1: Mean=0.0422, Bootstrap SE=0.0014
  level2: Mean=0.1021, Bootstrap SE=0.0051
  level3: Mean=0.1350, Bootstrap SE=0.0099


# **Visualizations**

In [24]:

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, metric, title in zip(axes,
    ["clip_score", "approx_novelty", "prompt_diversity"],
    ["CLIP Score", "Novelty", "Diversity"]):
    sns.boxplot(data=df, x="level", y=metric, ax=ax, palette="Set2")
    ax.set_title(f"{title} Distribution")
    ax.set_xlabel("Abstraction Level")
    ax.set_ylabel(metric)

plt.suptitle("Distribution of Metrics Across Abstraction Levels", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig("results_boxplots.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved: results_boxplots.png")


/tmp/ipython-input-1120/3789953.py:6: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=df, x="level", y=metric, ax=ax, palette="Set2")
/tmp/ipython-input-1120/3789953.py:6: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=df, x="level", y=metric, ax=ax, palette="Set2")
/tmp/ipython-input-1120/3789953.py:6: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=df, x="level", y=metric, ax=ax, palette="Set2")


Saved: results_boxplots.png


In [25]:
# Violin plots
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, metric, title in zip(axes,
    ["clip_score", "approx_novelty", "prompt_diversity"],
    ["CLIP Score", "Novelty", "Diversity"]):
    sns.violinplot(data=df, x="level", y=metric, inner="quartile", ax=ax, palette="Set2")
    ax.set_title(f"{title} Distribution")
    ax.set_xlabel("Abstraction Level")

plt.suptitle("Metric Distributions with Quartiles", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig("results_violins.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved: results_violins.png")


/tmp/ipython-input-1120/2655674007.py:7: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(data=df, x="level", y=metric, inner="quartile", ax=ax, palette="Set2")
/tmp/ipython-input-1120/2655674007.py:7: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(data=df, x="level", y=metric, inner="quartile", ax=ax, palette="Set2")
/tmp/ipython-input-1120/2655674007.py:7: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(data=df, x="level", y=metric, inner="quartile", ax=ax, palette="Set2")


Saved: results_violins.png


In [26]:
# Bar chart with 95% CI
def plot_mean_ci(ax, metric, title):
    summary = df.groupby("level")[metric].agg(["mean", "std", "count"]).reset_index()
    summary["ci95"] = 1.96 * (summary["std"] / np.sqrt(summary["count"]))

    ax.bar(summary["level"], summary["mean"], yerr=summary["ci95"],
           capsize=6, color=sns.color_palette("Set2", 3), edgecolor="black")
    ax.set_title(title)
    ax.set_ylabel("Mean Score")
    ax.set_xlabel("Abstraction Level")

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
plot_mean_ci(axes[0], "clip_score", "Mean CLIP Score ±95% CI")
plot_mean_ci(axes[1], "approx_novelty", "Mean Novelty ±95% CI")
plot_mean_ci(axes[2], "prompt_diversity", "Mean Diversity ±95% CI")

plt.suptitle("Mean Scores with 95% Confidence Intervals", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig("results_bar_ci.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved: results_bar_ci.png")


Saved: results_bar_ci.png


In [27]:
# Trend lines across levels
mean_df = df.groupby("level")[["clip_score", "approx_novelty", "prompt_diversity"]].mean().reset_index()

plt.figure(figsize=(7, 5))
for metric, label, color in zip(
    ["clip_score", "approx_novelty", "prompt_diversity"],
    ["CLIP Score", "Novelty", "Diversity"],
    ["steelblue", "coral", "mediumseagreen"]
):
    plt.plot(mean_df["level"], mean_df[metric], marker="o", label=label, color=color, linewidth=2)

plt.title("Metric Trends Across Abstraction Levels")
plt.xlabel("Abstraction Level")
plt.ylabel("Mean Score")
plt.legend()
plt.tight_layout()
plt.savefig("results_trends.png", dpi=150)
plt.close()
print("Saved: results_trends.png")


Saved: results_trends.png
